In [17]:
import pandas as pd
df = pd.read_csv("../data/nba_pbp_2020_2025_fe.csv")
df.head()

,Unnamed: 0,period,scoreHome,scoreAway,location,home_team_won,seconds,actionType_Ejection,actionType_Foul,actionType_Free Throw,actionType_Instant Replay,actionType_Jump Ball,actionType_Made Shot,actionType_Missed Shot,actionType_Rebound,actionType_Substitution,actionType_Timeout,actionType_Turnover,actionType_Violation,actionType_period
0,0,1,0.0,0.0,0.0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1
1,1,1,0.0,0.0,2.0,1,17,0,0,0,0,0,0,1,0,0,0,0,0,0
2,2,1,0.0,0.0,1.0,1,20,0,0,0,0,0,0,0,1,0,0,0,0,0
3,3,1,0.0,0.0,1.0,1,37,0,0,0,0,0,0,1,0,0,0,0,0,0
4,4,1,0.0,0.0,2.0,1,41,0,0,0,0,0,0,0,1,0,0,0,0,0


In [18]:
df = df.drop(columns = ['Unnamed: 0', 'actionType_Instant Replay', 'actionType_period', "actionType_Timeout", "actionType_Substitution", "actionType_Jump Ball", "actionType_Ejection"])
df["score_diff"] = df["scoreHome"] - df["scoreAway"]
df.head(20)

,period,scoreHome,scoreAway,location,home_team_won,seconds,actionType_Foul,actionType_Free Throw,actionType_Made Shot,actionType_Missed Shot,actionType_Rebound,actionType_Turnover,actionType_Violation,score_diff
0,1,0.0,0.0,0.0,1,0,0,0,0,0,0,0,0,0.0
1,1,0.0,0.0,2.0,1,17,0,0,0,1,0,0,0,0.0
2,1,0.0,0.0,1.0,1,20,0,0,0,0,1,0,0,0.0
3,1,0.0,0.0,1.0,1,37,0,0,0,1,0,0,0,0.0
4,1,0.0,0.0,2.0,1,41,0,0,0,0,1,0,0,0.0
5,1,2.0,0.0,2.0,1,45,0,0,1,0,0,0,0,2.0
6,1,2.0,0.0,1.0,1,62,0,0,0,1,0,0,0,2.0
7,1,2.0,0.0,1.0,1,65,0,0,0,0,1,0,0,2.0
8,1,2.0,3.0,1.0,1,71,0,0,1,0,0,0,0,-1.0
9,1,2.0,3.0,2.0,1,85,0,0,0,0,0,1,0,-1.0


In [14]:
params = {'max_depth': 6,
 'min_child_weight': 17,
 'subsample': 0.5643733174447721,
 'colsample_bytree': 0.9827635334946612,
 'learning_rate': 0.05072478407219559,
 'n_estimators': 771,
 'reg_alpha': 7,
 'reg_lambda': 10}

In [8]:
from sklearn.model_selection import train_test_split
X = df.drop(columns = ["home_team_won"])
y = df["home_team_won"]

X_train, X_test, y_train, y_test  = train_test_split(X, y, test_size = 0.2)

In [15]:
# import the model we want to first use
from xgboost import XGBClassifier

# import the metrics we wanna use
from sklearn.metrics import log_loss, roc_auc_score, accuracy_score

# track the models we want to use on our data set
import mlflow

# start an mlflow run
with mlflow.start_run():
    
    # initialize the model and use the best parameters we got and unpack them using **
    model = XGBClassifier(**params)
    
    # train the model
    xg = model.fit(X_train, y_train)

    # log the parameters in this case the default ones
    mlflow.log_params(model.get_params())

    # log the model and give it a name
    model_info = mlflow.xgboost.log_model(xg, name = "exp_xgboost_model2")
    
    # get the model predicted values
    y_pred = model.predict(X_test)

    # get the probabilite values, so we can use for log loss and roc_auc
    y_proba = model.predict_proba(X_test)

    # get the models metrics
    score = accuracy_score(y_test, y_pred)
    ll = log_loss(y_test, y_proba[:, 1])
    roc_auc = roc_auc_score(y_test, y_proba[:, 1])

    # log the metrics
    mlflow.log_metrics({"log_loss": ll, "accuracy_score": score, "roc_auc": roc_auc})